# Practical Session 04d &mdash; Beyond two classes: softmax regression

**Companion to Lecture&nbsp;04.** The last of four notebooks. Real problems often have more than two
classes (digits 0&ndash;9, species, topics). **Softmax regression** (multinomial logistic
regression) generalizes the sigmoid: one linear score per class, squashed into a probability
**distribution** over classes. We build softmax and its cross-entropy from scratch, check the
gradient, show that binary logistic is just the two-class case, then fit multiclass models &mdash;
including the **digits** dataset.

| # | Notebook | Topic |
|---|----------|-------|
| 04a | the model | the sigmoid, odds/log-odds, the decision boundary |
| 04b | training from scratch | cross-entropy, the gradient, gradient descent, convexity, $L_2$ |
| 04c | evaluation | confusion matrix, precision/recall/F1, ROC/AUC, thresholds, imbalance |
| **04d** | **multiclass** | **softmax from scratch, cross-entropy, decision regions, digits (this notebook)** |

> ⭐ **Key idea.** Give each class $k$ its own score $z_k=\mathbf{w}_k^\top\mathbf{x}+b_k$, then
> **softmax** turns the vector of scores into probabilities that are positive and sum to 1. The loss
> is again cross-entropy, and its gradient keeps the clean form $\mathbf{p}-\mathbf{y}$ (with a
> one-hot $\mathbf{y}$). Boundaries stay **linear**.

*Run each cell (Shift+Enter). Self-contained and offline.*

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from nb_utils import use_style
use_style("notes")

def sigmoid(z):
    z = np.asarray(z, float); out = np.empty_like(z)
    pos, neg = z >= 0, z < 0
    out[pos] = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[neg]); out[neg] = ez / (1.0 + ez)
    return out

print("ready")

## 1. The softmax function (from scratch)

For a vector of $K$ scores $\mathbf{z}=(z_1,\dots,z_K)$,

$$\text{softmax}(\mathbf{z})_k=\frac{e^{z_k}}{\sum_{j=1}^{K}e^{z_j}}.$$

Every output is positive and they **sum to 1** &mdash; a probability distribution over the classes.
The naive formula overflows for large scores, so we subtract the row max first (this cancels out
and changes nothing mathematically).

In [ ]:
def softmax(Z):
    "Numerically stable softmax along the last axis (works on a vector or a batch)."
    Z = np.asarray(Z, float)
    Z = Z - Z.max(axis=-1, keepdims=True)      # stability: subtract the max
    e = np.exp(Z)
    return e / e.sum(axis=-1, keepdims=True)

z = np.array([2.0, 1.0, -1.0])
p = softmax(z)
print("scores :", z)
print("softmax:", p.round(4), " sum =", p.sum().round(6))
print("stable for huge scores:", softmax(np.array([1000.0, 999.0, 998.0])).round(4))

## 2. One score per class; predict the argmax

Stack the per-class weight vectors into a matrix $\mathbf{W}$ (one column per class) and biases
$\mathbf{b}$. The scores are $\mathbf{z}=\mathbf{x}\mathbf{W}+\mathbf{b}$, the probabilities are
$\text{softmax}(\mathbf{z})$, and the prediction is the **most probable class**, $\arg\max_k p_k$.

In [ ]:
from sklearn.datasets import make_blobs
X, y = make_blobs(n_samples=300, centers=[(-2.4, -1.2), (2.4, -1.2), (0.0, 2.6)],
                  cluster_std=1.1, random_state=3)
K = 3

def predict_proba(X, W, b): return softmax(X @ W + b)
def predict(X, W, b):       return predict_proba(X, W, b).argmax(axis=1)

# a random (untrained) W just to show the shapes and the argmax rule
rng = np.random.default_rng(0)
W0, b0 = rng.normal(size=(2, K)), np.zeros(K)
P = predict_proba(X[:3], W0, b0)
print("first 3 probability rows (each sums to 1):")
print(P.round(3), " row sums:", P.sum(1).round(3))
print("predicted classes:", predict(X[:3], W0, b0))

## 3. Cross-entropy and its gradient (from scratch)

With the true class one-hot encoded as $\mathbf{y}_i$, the loss is
$J=-\tfrac1n\sum_i\sum_k y_{ik}\log p_{ik}$ &mdash; the same cross-entropy, now summed over classes.
Its gradient keeps the clean "prediction minus truth" form:

$$\frac{\partial J}{\partial \mathbf{W}}=\frac1n\,\mathbf{X}^\top(\mathbf{P}-\mathbf{Y}),\qquad
\frac{\partial J}{\partial \mathbf{b}}=\overline{\mathbf{P}-\mathbf{Y}}.$$

We implement it and check against a numerical gradient.

In [ ]:
def onehot(y, K): return np.eye(K)[y]

def ce_loss(W, b, X, y, eps=1e-12):
    P = softmax(X @ W + b)
    return -np.mean(np.log(P[np.arange(len(y)), y] + eps))

def ce_grad(W, b, X, y):
    n = len(y)
    P = softmax(X @ W + b)
    G = (P - onehot(y, W.shape[1])) / n
    return X.T @ G, G.sum(axis=0)

W = rng.normal(size=(2, K)) * 0.3; b = np.zeros(K)
gW, gb = ce_grad(W, b, X, y)

# numerical gradient for one weight entry and one bias
h = 1e-6
i, j = 1, 2
Wp = W.copy(); Wp[i, j] += h; Wm = W.copy(); Wm[i, j] -= h
gW_num = (ce_loss(Wp, b, X, y) - ce_loss(Wm, b, X, y)) / (2 * h)
print(f"analytic dJ/dW[{i},{j}] = {gW[i,j]:.6f}   numeric = {gW_num:.6f}   match: {np.isclose(gW[i,j], gW_num, atol=1e-5)}")

## 4. Binary logistic *is* two-class softmax

Softmax with two classes reduces to the sigmoid: with scores $(0, z)$,
$\text{softmax}(0,z)=\bigl(\tfrac{1}{1+e^{z}},\,\tfrac{e^{z}}{1+e^{z}}\bigr)=(1-\sigma(z),\,\sigma(z))$.
So the binary model of 04a/04b is just the $K=2$ special case.

In [ ]:
z = np.linspace(-6, 6, 200)
soft = softmax(np.column_stack([np.zeros_like(z), z]))     # softmax of (0, z), per row
print("softmax(0, z)[:,1] == sigmoid(z):", np.allclose(soft[:, 1], sigmoid(z)))
print("softmax(0, z)[:,0] == 1 - sigmoid(z):", np.allclose(soft[:, 0], 1 - sigmoid(z)))

## 5. Train softmax regression; the decision regions

Now fit the 3-class blobs by gradient descent (with a whiff of $L_2$), and draw the regions. Each
pairwise boundary is a **straight line**; the three meet at a point, carving the plane into three
convex pieces. We check the accuracy against scikit-learn's multinomial `LogisticRegression`.

In [ ]:
def softmax_fit(X, y, K, lr=0.5, n_iter=3000, lam=1e-3):
    n, d = X.shape
    W, b = np.zeros((d, K)), np.zeros(K)
    for _ in range(n_iter):
        gW, gb = ce_grad(W, b, X, y)
        W -= lr * (gW + lam * W)
        b -= lr * gb
    return W, b

W, b = softmax_fit(X, y, K)
acc_scratch = (predict(X, W, b) == y).mean()

from sklearn.linear_model import LogisticRegression
sk = LogisticRegression().fit(X, y)      # multinomial by default
print(f"our softmax GD accuracy : {acc_scratch:.3f}")
print(f"sklearn accuracy        : {sk.score(X, y):.3f}")

# decision regions from the from-scratch model
pad = 1.0
xx, yy = np.meshgrid(np.linspace(X[:,0].min()-pad, X[:,0].max()+pad, 300),
                     np.linspace(X[:,1].min()-pad, X[:,1].max()+pad, 300))
grid = np.column_stack([xx.ravel(), yy.ravel()])
Z = predict(grid, W, b).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(5.6, 4.4))
ax.contourf(xx, yy, Z, levels=[-0.5,0.5,1.5,2.5], colors=["#0072B2","#D55E00","#009E73"], alpha=0.22)
ax.contour(xx, yy, Z, levels=[0.5, 1.5], colors="0.3", linewidths=1.2)
for k, col in enumerate(["#0072B2", "#D55E00", "#009E73"]):
    ax.scatter(X[y==k,0], X[y==k,1], s=18, color=col, edgecolor="white", lw=0.3, label=f"class {k}")
ax.set_xlabel("$x_1$"); ax.set_ylabel("$x_2$"); ax.set_title("softmax: three classes, linear boundaries")
ax.legend(fontsize=8, loc="upper right"); plt.show()

## 6. A real multiclass problem: handwritten digits

The `digits` dataset is 1797 images of $8\times8$ pixels, ten classes (0&ndash;9). Softmax
regression on the raw pixels already does well. We scale, fit, and read the **confusion matrix** to
see which digits get mixed up.

In [ ]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, accuracy_score

dig = load_digits()
Xtr, Xte, ytr, yte = train_test_split(dig.data, dig.target, test_size=0.3, random_state=0, stratify=dig.target)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)).fit(Xtr, ytr)
pred = model.predict(Xte)
print(f"test accuracy: {accuracy_score(yte, pred):.3f}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(9.0, 4.0))
a1.imshow(dig.images[0], cmap="gray_r"); a1.set_title(f"an example (label {dig.target[0]})")
a1.set_xticks([]); a1.set_yticks([])
# confusion matrix
M = confusion_matrix(yte, pred)
im = a2.imshow(M, cmap="Blues"); a2.set_title("confusion matrix (10 classes)")
a2.set_xlabel("predicted"); a2.set_ylabel("true")
a2.set_xticks(range(10)); a2.set_yticks(range(10))
fig.colorbar(im, ax=a2, shrink=0.8); plt.show()
print("most confused pairs (off-diagonal):")
Moff = M.copy(); np.fill_diagonal(Moff, 0)
for _ in range(3):
    i, j = np.unravel_index(Moff.argmax(), Moff.shape)
    print(f"  true {i} -> predicted {j}: {Moff[i,j]} times"); Moff[i, j] = 0

## Recap & exercises

**Recap.**
* **Softmax** turns $K$ per-class scores into a probability distribution ($\propto e^{z_k}$, sums to 1); use the **max-subtraction** trick for stability.
* The multiclass model predicts $\arg\max_k p_k$; boundaries stay **linear** (convex regions).
* The loss is **cross-entropy** $-\tfrac1n\sum_i\log p_{i,y_i}$; its gradient is again
  $\tfrac1n\mathbf{X}^\top(\mathbf{P}-\mathbf{Y})$ with one-hot $\mathbf{Y}$ &mdash; checked numerically.
* **Binary logistic is the $K=2$ softmax**: $\text{softmax}(0,z)=(1-\sigma(z),\sigma(z))$.
* Our from-scratch softmax GD matches scikit-learn, and softmax regression reads handwritten **digits** well.

**Exercises.**
1. Add $L_2$ strength `lam=0.1` in `softmax_fit`. How do the boundaries and accuracy change?
2. Softmax is **over-parameterized** (adding a constant to every score leaves it unchanged). Verify: `softmax(z) == softmax(z + 5)`. Why is this harmless?
3. On digits, show three **misclassified** test images with their predicted vs true labels. Are they genuinely ambiguous?
4. Compare `LogisticRegression(multi_class="ovr")` (one-vs-rest) to the default multinomial on the blobs. Do the boundaries differ?
5. Replace the raw pixels with `StandardScaler` **off**. Does accuracy or convergence change? (Watch for a convergence warning.)

*That completes the Session&nbsp;04 practical arc:* the model (04a), training it (04b), judging it
(04c), and extending it to many classes (04d) &mdash; all from scratch and checked against scikit-learn.